# Foundation 03 — Security Invariants and Blast Radius

Translate security objectives into state, transition, and temporal invariants; enforce them outside the model; and measure blast radius as an explicit vector. The credential-free lab proves local safety and utility properties without presenting property tests or simulations as production proof.

![Security invariants from objectives to bounded effects](architecture.svg)

An untrusted proposal enters a trusted invariant gate with authenticated identity, current capability, policy, budgets, approval, idempotency, and kill-switch state. Decisions and effects become attributable trajectory evidence.

## 1. Load the course lab

The notebook imports the reusable course module rather than copying its security logic.

In [ ]:
import runpy
from datetime import datetime, timedelta, timezone
from dataclasses import replace
import sys
sys.path.insert(0, '.')
ns = runpy.run_path('lab.py')
ActionProposal, Operation, DecisionStatus, PolicyLimits = (ns[name] for name in ('ActionProposal','Operation','DecisionStatus','PolicyLimits'))
build_runtime, valid_refund, execute_approved = (ns[name] for name in ('build_runtime','valid_refund','execute_approved'))
now = datetime(2026,9,21,12,0,tzinfo=timezone.utc)
engine, actor, reviewer, grant = build_runtime(now=now)

## 2. Establish the safe baseline

Observe the trusted inputs and the decision evidence before injecting failures.

In [ ]:
proposal = valid_refund('notebook:refund:1',2500)
allowed = execute_approved(engine,actor,reviewer,proposal,grant,approval_id='approval:notebook:1',now=now)
audit = ns['audit_trajectory'](engine)
assert allowed.effect_applied and audit.accepted
{'decision': allowed, 'state': {'writes': engine.state.writes_used, 'amount_cents': engine.state.amount_used_cents}, 'audit': audit}

## 3. Inject an attack

Change one security-relevant boundary and keep the rest of the fixture stable.

In [ ]:
attack_engine, attack_actor, _, attack_grant = build_runtime(now=now)
cross_tenant = ActionProposal(Operation.ISSUE_REFUND,'case:south:9','notebook:attack',100,claimed_subject='admin',claimed_tenant='south')
baseline_accepts = ns['unsafe_text_only_baseline'](cross_tenant)
controlled = attack_engine.execute(attack_actor,cross_tenant,grant_id=attack_grant.grant_id,now=now)
assert baseline_accepts and controlled.reason == 'tenant-isolation' and not controlled.effect_applied
{'text_only_baseline': baseline_accepts, 'controlled': controlled}

## 4. Attempt a bypass

The assertions below make the security property executable and regression-testable.

In [ ]:
duplicate = engine.execute(actor,proposal,grant_id=grant.grant_id,approval_id='approval:notebook:1',now=now)
collision = engine.execute(actor,replace(proposal,amount_cents=2600),grant_id=grant.grant_id,now=now)
engine.activate_kill_switch(at=now+timedelta(seconds=1))
after_kill = engine.execute(actor,valid_refund('notebook:refund:2',100),grant_id=grant.grant_id,now=now+timedelta(seconds=1))
assert duplicate.status is DecisionStatus.DUPLICATE and not duplicate.effect_applied
assert collision.reason == 'operation-id-collision'
assert after_kill.reason == 'kill-switch'
(duplicate,collision,after_kill)

## 5. Evaluate observable outcomes

Use explicit denominators or counts. Private model reasoning is neither required nor recorded.

In [ ]:
report,cases = ns['evaluate_controls'](now=now)
assert (report.cases,report.valid_cases,report.attack_cases,report.failure_cases) == (12,3,7,2)
assert report.attack_effect_rate == report.valid_task_block_rate == 0
assert report.attack_block_rate == report.valid_task_success_rate == 1
{'populations': {'all': report.cases, 'valid': report.valid_cases, 'attack': report.attack_cases, 'failure': report.failure_cases}, 'rates': {'attack_effect': report.attack_effect_rate, 'attack_block': report.attack_block_rate, 'valid_success': report.valid_task_success_rate, 'valid_block': report.valid_task_block_rate, 'trace_completeness': report.trace_completeness_rate}, 'outcomes': {case.name: (case.decision.status.value,case.decision.reason) for case in cases}}

## 6. Exercise a second failure mode

In [ ]:
approval_down,down_actor,_,down_grant = build_runtime(now=now,approval_available=False)
failed = approval_down.execute(down_actor,valid_refund('notebook:failure',100),grant_id=down_grant.grant_id,approval_id='approval:any',now=now)
assert failed.status is DecisionStatus.ERROR and failed.reason == 'approval-service-unavailable'
assert not failed.effect_applied and not approval_down.state.effects
failed

## 7. Blast radius is a vector, not a score

Compare maximum tenants, resources, write operations, counts, value, egress destinations, and credential lifetime separately. A smaller value in one dimension does not compensate for an unacceptable expansion in another.

In [ ]:
profiles = ns['blast_radius_profiles'](now=now)
assert profiles['broad'].tenants == 3 and profiles['bounded'].tenants == 1
assert profiles['broad'].resources > profiles['bounded'].resources > profiles['deny_all'].resources
profiles

## 8. Atomic budget reservation under concurrency

Eight individually approved actions compete for a one-write budget. The process lock is a teaching analogue for one shared production consistency boundary.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
limits = PolicyLimits(max_writes=1,max_total_cents=1000,max_effect_cents=1000)
race,race_actor,race_reviewer,race_grant = build_runtime(now=now,limits=limits)
proposals = [valid_refund(f'notebook:race:{i}',1000) for i in range(8)]
for i,item in enumerate(proposals): race.approvals.issue(item,race_actor,race_reviewer,approval_id=f'approval:race:{i}',now=now)
def invoke(pair):
 i,item=pair
 return race.execute(race_actor,item,grant_id=race_grant.grant_id,approval_id=f'approval:race:{i}',now=now)
with ThreadPoolExecutor(max_workers=8) as pool: race_decisions=list(pool.map(invoke,enumerate(proposals)))
assert sum(item.effect_applied for item in race_decisions)==1
assert sum(item.reason=='write-budget' for item in race_decisions)==7
[(item.status.value,item.reason) for item in race_decisions]

## 9. Hypothesis 6: generated invariant checks

The real library runs deterministic generated examples for tenant isolation, budget sequences, and claimed identity. A missing counterexample is evidence about the encoded strategies, not a proof of completeness or production equivalence.

In [ ]:
hypothesis_lab = runpy.run_path('hypothesis_adapter.py')
property_report = hypothesis_lab['run_properties']()
assert property_report == {'properties':3,'generated_examples_per_property':75}
property_report

## 10. Production replacement

Production replacement: authenticated human and workload identity; signed or server-resolved short-lived capabilities; durable exact approval with separation of duties; transactional budget, idempotency, and effect ledgers; provider reconciliation for unknown outcomes; policy distribution with integrity, activation status, freshness SLO, and rollback; independently operated kill switches with verified propagation; hard tenant/account/network/runtime isolation; protected OpenTelemetry and audit evidence; continuous blast-radius inventory; risk-derived property/state-machine tests; and formal modeling for high-consequence concurrent protocols. The local lock and generated examples prove only the encoded in-process fixture.

## 11. Exercises

1. Add a destination quota and update the blast-radius vector.
2. Add capability revocation and prove no new effect starts afterward.
3. Extend Hypothesis with a rule-based state machine for issue, duplicate, collision, revoke, and kill.
4. Design safe read-only degradation for an approval-service outage.
5. Specify the budget/idempotency transition in TLA+ or PlusCal and explain model-to-code drift.
6. Define an OPA or Cedar integration contract with authenticated inputs, undefined/error semantics, decision evidence, and final application enforcement.

## Checkpoint

Explain which trusted component enforces the invariant, what evidence proves the decision, and what residual risk remains.